In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import librosa
import numpy as np
import time
import optuna
from optuna.trial import Trial
import joblib

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *
from sj_utils.collection_utils import SafetyDict

In [ ]:
from rt_whisper import streamers
from rt_whisper.data import Param, Result

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/train/"
STUDY = "/workspaces/dev/study/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
src = Path(SOURCE)
study_path = Path(STUDY) / "study.pkl"
study_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def objective(trial:Trial):
    hyperparameters = SafetyDict({
        "weighted_and_offset_token_boundary": trial.suggest_int(
            "weighted_and_offset_token_boundary", 4000, 16000, step=100
        ),
        "duration_filter_z":{
            "default": 2.0,
            "en": trial.suggest_float("duration_filter_z", 3.0, 5.0, step = 0.1)
        },
        "probability_filter":{
            "z":{
                "default": 3.0,
                "en": trial.suggest_float("probability_filter_z", 3.0, 5.0, step = 0.1)
            },
            "min_prob": {
                "default": 1.0,
                "en": trial.suggest_float("probability_filter_min_prob", 0, 1, step = 0.05)
            },
        },
        "selector":{
            "search_range_sc": {
                "default": 24000,
                "en": trial.suggest_int("search_range_sc", 0, 48000, step=1000)
            },
            "threshold":{
                "default": 0.5,
                "en": trial.suggest_float("threshold", 0, 1, step=0.05)
            },
            "padding": {
                "default": 3200,
                "en": trial.suggest_int("padding", 0, 18000, step=100)
            },
            "tolerance": {
                "default": 8000,
                "en": trial.suggest_int("tolerance", 0, 16000, step=100)
            }
        },
        "max_overlap_duration": 96000
    })

    token_streamer = streamers.get_token_streamer_with_vad_v2(hyperparameter=hyperparameters)

    def transcriber(flac:Path) -> TRNFormat:
        audio, _ = librosa.load(flac, sr=SAMPLE_RATE)

        completed = []
        param = Param()
        # start_time = time.perf_counter()
        for segment in segment_audio(audio):
            param.chunk = segment
            param.language="en"
            result:Result = token_streamer.process(param)
            completed.extend(result.completed)
            param.update(result)
        completed.extend(result.candidate)
        # end_time = time.perf_counter()
        # print(f"Processed {flac.stem} in {end_time - start_time:.2f} seconds")

        return TRNFormat(
            id = flac.stem,
            text = normalize_text_only_en(
                " ".join([s.text for s in completed])
            ).upper()
        )

    data = search_all_ref_and_hyp(src, transcriber, 1)

    concat_result = {}
    for value in data.values():
        for k, v in value.items():
            if k not in concat_result:
                concat_result[k] = []
            concat_result[k].extend(v)

    output = sclite_trn(concat_result["ref"], concat_result["hyp"])
    result = parse_sclite_summary(output)

    return result["wer_percent"]

In [ ]:
if study_path.exists():
    study = joblib.load(study_path)
else:
    study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)

In [ ]:
study.best_value

In [ ]:
study.best_params

In [ ]:
joblib.dump(study, study_path)